# AISA-ArabicFC — **Qwen3-4B** QLoRA fine-tune (Colab / Kaggle)

Team **Ṣaqr (صقر)**.  Alternative model to Gemma — strong Arabic + native tool-use.

**Runtime:** L4 / A100 (bf16) *or* T4 / P100 (fp16, auto). Qwen3 has **no** bf16 requirement, so it also runs on Kaggle T4.

**Base:** `unsloth/Qwen3-4B-Instruct-2507` (QLoRA / 4-bit). Fallback id: `unsloth/Qwen3-4B`.

**Design — "safe port":** we reuse the dataset's OWN `text` field (the `<start_function_call>…` DSL) and the same parser + scorer as the Gemma notebook. Qwen just learns the format by fine-tuning. Only the base model + a couple of Qwen-specific tweaks change. The custom markers are plain text to Qwen, so we add `stop_strings=["<end_of_turn>"]` at inference and strip the literal `<bos>`.

Score to beat: **0.8721 Overall A** (dev leaderboard).

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth unsloth_zoo   # local / non-Colab
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
import torch; torch._dynamo.config.recompile_limit = 64

In [ ]:
import unsloth, transformers, datasets, trl
print("unsloth", unsloth.__version__)       # 2026.7.2
print("transformers", transformers.__version__)  # 5.5.0
print("datasets", datasets.__version__)     # 4.3.x
print("trl", trl.__version__)               # 0.24.0

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.7.6
transformers 5.5.0
datasets 4.0.0
trl 1.9.2


In [ ]:
# 0) RUN CONFIG — the ONLY cell you normally edit.
#    Every run is keyed by RUN, so a different LoRA setting (or a different person)
#    NEVER shares a checkpoint folder with a previous run. Set LoRA here, not below.
LORA_R     = 16     # LoRA rank
LORA_ALPHA = 16     # LoRA alpha (scaling)
EPOCHS     = 2      # 2 = sweet spot on this task (3 overfits, see project notes)

RUN = f"r{LORA_R}a{LORA_ALPHA}e{EPOCHS}"
print("RUN =", RUN, "  (checkpoints, adapter and merged model are all namespaced by this)")


RUN = r16a16e2   (checkpoints, adapter and merged model are all namespaced by this)


In [ ]:
# 2) Load BOTH splits + parser  (reuse dataset's own DSL text; port to Qwen)
import re, json, unicodedata
from datasets import load_dataset

# Qwen isn't Gemma, so the dataset's <bos>/<start_of_turn> are PLAIN TEXT here.
# That's fine: train + inference use the SAME text, so Qwen learns the format.
# We only strip the literal leading "<bos>" (Qwen has its own BOS handling).
def prep(text):
    return text[len("<bos>"):] if text.startswith("<bos>") else text

SEP = "<start_of_turn>model"   # dataset's turn separator (plain-text marker for Qwen)

train_raw = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="train")
dev_raw   = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="dev")
train_raw = train_raw.map(lambda r: {"text": prep(r["text"])})
dev_raw   = dev_raw.map(lambda r: {"text": prep(r["text"])})
print("train:", len(train_raw), " dev:", len(dev_raw))

def parse_model_output(text):
    out = {"function_name": "none", "arguments": {}, "think": ""}
    m = re.search(r"<think>\s*(.*?)\s*</think>", text, re.DOTALL)
    if m: out["think"] = m.group(1).strip()
    m = re.search(r"<start_function_call>\s*call:(\w+)\{(.*?)\}\s*<end_function_call>", text, re.DOTALL)
    if m:
        out["function_name"] = m.group(1)
        for key, sval, nval in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", m.group(2)):
            val = sval if sval else nval
            try: val = float(val) if "." in str(val) else int(val)
            except (ValueError, TypeError): pass
            out["arguments"][key] = val
    return out

def user_query(text):
    m = re.search(r"<start_of_turn>user\s*(.*?)\s*<end_of_turn>", text, re.DOTALL)
    return m.group(1).strip() if m else ""

README.md:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  678kB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating dev split:   0%|          | 0/545 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1125 [00:00<?, ? examples/s]

Map:   0%|          | 0/10550 [00:00<?, ? examples/s]

Map:   0%|          | 0/545 [00:00<?, ? examples/s]

train: 10550  dev: 545


In [ ]:
# 3) OFFICIAL scorer — the real leaderboard grader, pulled from the HF Space.
#    Replaces the old hand-rolled local scorer, which UNDERESTIMATED by ~0.045
#    and mis-ranked models (it lacked semantic normalisation: names احمد<->ahmed,
#    products آيفون<->iphone, dates الخميس<->Thursday/ISO, currencies, countries,
#    number-words, float-collapse, list-as-frozenset, optional-param strip).
import os, sys, subprocess, unicodedata

SPACE = "https://huggingface.co/spaces/TuwaiqAcademy/AISA-ArabicFC-SharedTask-Leaderboard/raw/main"
GRADER_DIR = "/content/official_scorer" if os.path.isdir("/content") else "./official_scorer"
os.makedirs(GRADER_DIR, exist_ok=True)
for _f in ("normalize.py", "eval_lib.py", "data_loader.py"):
    _p = os.path.join(GRADER_DIR, _f)
    if not os.path.exists(_p):
        subprocess.run(["curl", "-sfL", f"{SPACE}/{_f}", "-o", _p], check=True)
if GRADER_DIR not in sys.path:
    sys.path.insert(0, GRADER_DIR)

from eval_lib import evaluate          # official metric weights + per-dialect breakdown
from data_loader import load_gold      # official gold extraction (messages[].tool_calls)
from normalize import args_match       # official value normalisation / matching

# Official gold for dev: id == row index, same order as dev_raw.
GOLD = load_gold("dev")
assert len(GOLD) == len(dev_raw), f"gold {len(GOLD)} != dev_raw {len(dev_raw)}"

# Back-compat view used by the PP / error-analysis cells: [{"fn","args"}] by row order.
dev_gold = [{"fn": g["tool_called"], "args": g["arguments"]} for g in GOLD]

# ---- kept only because PP v2 (cell 12) and the error-analysis cell call them ----
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def norm(v):
    s = str(v).translate(AR2EN).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    for a, b in [("أ","ا"), ("إ","ا"), ("آ","ا"), ("ى","ي"), ("ة","ه")]:
        s = s.replace(a, b)
    try:
        f = float(s); s = str(int(f)) if f == int(f) else str(f)
    except ValueError:
        pass
    return s

def norm_args(d):
    return {norm(k): norm(v) for k, v in d.items()}

# Official per-row arg verdict — use THIS in error analysis so buckets agree with the score.
def arg_ok(pred, i):
    """True if prediction `pred` matches gold row `i` under the official matcher."""
    g = GOLD[i]
    return args_match(pred.get("arguments"), g["arguments"], g["tool_called"])
# --------------------------------------------------------------------------------

def score_full(predictions, gold=None):
    """Full official report dict (fnacc, argem, thinkrate, overall_a/b, per-dialect, gaps)."""
    gold = gold if gold is not None else GOLD
    preds = [{**p, "id": p.get("id", i)} for i, p in enumerate(predictions)]
    return evaluate(preds, gold)

def score(predictions):
    """Drop-in replacement for the old scorer: -> (FnAcc, ArgEM, OverallA).
    Full report of the last call is left in globals as LAST_REPORT."""
    global LAST_REPORT
    LAST_REPORT = score_full(predictions)
    return LAST_REPORT["fnacc"], LAST_REPORT["argem"], LAST_REPORT["overall_a"]

def report(predictions, label=""):
    r = score_full(predictions)
    print(f"--- {label or 'official scorer'} ---")
    print(f"FnAcc {r['fnacc']:.4f} | ArgEM {r['argem']:.4f} | ThinkRate {r['thinkrate']:.4f}")
    print(f"Overall A {r['overall_a']:.4f} | Overall B {r['overall_b']:.4f}")
    print(f"n={r['n_total']} (pos {r['n_positive']}, neg {r['n_negative']}), missing {r['missing']}")
    for d, s in sorted(r["dialect_breakdown"].items(), key=lambda kv: -kv[1]["n"]):
        print(f"  {d:10s} FnAcc {s['fnacc']:.3f}  ArgEM {s['argem']:.3f}  (n={s['n']}, pos={s['n_positive']})")
    print(f"gap_fnacc {r['gap_fnacc']:.4f} | gap_argem {r['gap_argem']:.4f}")
    return r

_npos = sum(1 for g in GOLD if g["requires_function"])
print(f"official scorer ready — dev gold {len(GOLD)} rows ({_npos} positive, {len(GOLD)-_npos} negative)")
print("use score(preds) -> (FnAcc, ArgEM, OverallA)  |  report(preds) for the full breakdown")


official scorer ready — dev gold 545 rows (500 positive, 45 negative)
use score(preds) -> (FnAcc, ArgEM, OverallA)  |  report(preds) for the full breakdown


In [ ]:
# 4) Load Qwen3-4B with Unsloth (QLoRA / 4-bit)
from unsloth import FastModel, is_bfloat16_supported
import torch

max_seq_length = 2048
BASE = "unsloth/Qwen3-4B-Instruct-2507"   # fallback: "unsloth/Qwen3-4B"

USE_BF16 = is_bfloat16_supported()        # L4/A100 -> True ; T4/P100 -> False (fp16)
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("bf16 supported:", USE_BF16, "-> compute dtype:", COMPUTE_DTYPE)
# NOTE: Qwen3 (unlike Gemma E-models) runs fine in fp16, so NO hard bf16 assert here.

model, tokenizer = FastModel.from_pretrained(
    model_name = BASE,
    max_seq_length = max_seq_length,
    dtype = COMPUTE_DTYPE,
    load_in_4bit = not USE_BF16,  # A100/L4 (bf16) -> 16-bit LoRA (better quality); T4/P100 -> 4-bit QLoRA
    full_finetuning = False,
)

bf16 supported: True -> compute dtype: torch.bfloat16
==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# 5) VERIFY BEFORE TRAINING — SEP present in the DATA, row head, prompt lengths
# (We train on the dataset's DSL text, so SEP lives in the row text, NOT Qwen's chat template.)
print("SEP in train row:", SEP in train_raw[0]["text"], "| SEP token ids:",
      getattr(tokenizer, "tokenizer", tokenizer)(SEP, add_special_tokens=False)["input_ids"])

print("\n=== dataset row (head) ===")
print(train_raw[0]["text"][:400])

txt_tok = getattr(tokenizer, "tokenizer", tokenizer)
lens = [len(txt_tok(r["text"], add_special_tokens=False)["input_ids"]) for r in dev_raw]
print("\nmax prompt+answer tokens:", max(lens), " (must be well under", max_seq_length, ")")

SEP in train row: True | SEP token ids: [27, 2468, 3575, 37274, 29, 2528]

=== dataset row (head) ===
<start_of_turn>developer
عند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>declaration:compare_prices{description:<escape>مقارنة أسعار منتج في متاجر مختلفة<escape>,parameters:{properties:{category:{description:<escape>الفئة<escape>,type:<es

max prompt+answer tokens: 944  (must be well under 2048 )


In [ ]:
# 6) LoRA (FastModel signature — text layers only)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R, lora_alpha = LORA_ALPHA, lora_dropout = 0.0,   # from the RUN CONFIG cell
    bias = "none",
    random_state = 3407,
)

In [ ]:
# 7) SFT dataset = the rows' OWN text
sft_train = train_raw.select_columns(["text"])
print(sft_train[0]["text"][:400], "\n...")

<start_of_turn>developer
عند الحاجة لاستدعاء أداة: اكتب أولا <think> reasoning قصير </think> ثم TOOL_CALL فقط. لا تقدم إجابة نهائية قبل استدعاء الأداة. إذا كان سؤال الطقس بلا يوم محدد فاعتبره اليوم (days=1).<start_function_declaration>declaration:compare_prices{description:<escape>مقارنة أسعار منتج في متاجر مختلفة<escape>,parameters:{properties:{category:{description:<escape>الفئة<escape>,type:<es 
...


In [ ]:
# ===== Google Drive: persist CHECKPOINTS + final model across sessions =====
# Run this BEFORE building the trainer. Colab mounts Drive; Kaggle falls back to
# /kaggle/working. CKPT_DIR holds resumable training checkpoints; DRIVE_DIR holds
# the final saved model.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/arabicfc"      # Colab -> Google Drive
except Exception:
    DRIVE_DIR = "/kaggle/working/arabicfc"             # Kaggle fallback
assert "RUN" in globals(), "Run the RUN CONFIG cell first."
#CKPT_DIR = f"{DRIVE_DIR}/qwen3_4b_{RUN}_checkpoints"   # namespaced -> no cross-run collisions
CKPT_DIR = f"{DRIVE_DIR}/qwen3_4b_r16a16e2_checkpoints"   # namespaced -> no cross-run collisions


os.makedirs(CKPT_DIR, exist_ok=True)
print("Drive dir   ->", DRIVE_DIR)
print("Checkpoints ->", CKPT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive dir   -> /content/drive/MyDrive/arabicfc
Checkpoints -> /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints


In [ ]:
# 8) Trainer with ROBUST manual response-masking (Gemma markers are plain text to Qwen,
#    so train_on_responses_only's marker search is unreliable — we mask by token id instead).
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

txt_tok  = getattr(tokenizer, "tokenizer", tokenizer)
RESP_IDS = txt_tok("<start_of_turn>model\n", add_special_tokens=False)["input_ids"]

def _last_sub(ids, sub):
    for i in range(len(ids) - len(sub), -1, -1):
        if ids[i:i+len(sub)] == sub:
            return i + len(sub)
    return None

def tok_and_mask(ex):
    ids = txt_tok(ex["text"], add_special_tokens=False,
                  truncation=True, max_length=max_seq_length)["input_ids"]
    cut = _last_sub(ids, RESP_IDS)              # train only AFTER the model turn opens
    labels = [-100] * len(ids)
    if cut is not None:
        labels[cut:] = ids[cut:]                # unmask the answer (think + call)
    return {"input_ids": ids, "attention_mask": [1]*len(ids), "labels": labels}

sft_train = train_raw.map(tok_and_mask, remove_columns=train_raw.column_names,
                          desc="tokenize+mask")

# quick coverage check: how many rows actually got an answer span?
_no_span = sum(1 for r in sft_train.select(range(min(500, len(sft_train))))
               if all(l == -100 for l in r["labels"]))
print("rows (of 500 sampled) with NO answer span:", _no_span, "(should be 0)")

collator = DataCollatorForSeq2Seq(tokenizer=txt_tok, padding=True, label_pad_token_id=-100)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = sft_train,        # already tokenized + masked
    data_collator = collator,
    args = SFTConfig(
        max_length = max_seq_length,
        per_device_train_batch_size = 4 if USE_BF16 else 2,   # A100 -> 4, T4 -> 2
        gradient_accumulation_steps = 4 if USE_BF16 else 8,   # effective batch = 16 either way
        warmup_steps = 20,
        bf16 = USE_BF16,
        fp16 = not USE_BF16,
        num_train_epochs = EPOCHS,   # RUN CONFIG cell
        learning_rate = 2e-4,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = CKPT_DIR,          # -> Google Drive (survives disconnects)
        save_strategy = "steps",
        save_steps = 250,               # checkpoint every 250 steps
        save_total_limit = 3,           # keep newest 3 (Drive space)
        report_to = "none",
        padding_free = False,   # we pad via DataCollatorForSeq2Seq (not packing)
        dataset_kwargs = {"skip_prepare_dataset": True},  # data is pre-tokenized, don't re-process
    ),
)
# NOTE: no train_on_responses_only — masking is already applied above.

rows (of 500 sampled) with NO answer span: 0 (should be 0)


NameError: name 'model' is not defined

In [ ]:
# 9) MASKING SANITY — must print ONLY the answer (think + call), no prompt/tools.
# If you see tool declarations here, the marker strings are wrong -> STOP, do not train.
row = trainer.train_dataset[0]
dec = tokenizer.decode([tok for tok, lab in zip(row["input_ids"], row["labels"]) if lab != -100])
print(dec)

NameError: name 'trainer' is not defined

In [ ]:
# 10) Train — resumes ONLY a checkpoint from this exact LoRA config, then saves to Drive.
import os, json
from transformers.trainer_utils import get_last_checkpoint

_last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None

# GUARD — never resume a checkpoint trained with different LoRA settings.
# Changing r raises a shape error (loud). Changing ONLY lora_alpha does not: the
# shapes match, global_step is restored, and training "completes" having done
# zero steps. That silent no-op is what this check exists to prevent.
if _last:
    _cfg = json.load(open(os.path.join(_last, "adapter_config.json")))
    _r, _a = _cfg.get("r"), _cfg.get("lora_alpha")
    if (_r, _a) != (LORA_R, LORA_ALPHA):
        raise RuntimeError(
            f"STALE CHECKPOINT.\n  {_last}\n  was trained with r={_r}, alpha={_a}"
            f" but this run is r={LORA_R}, alpha={LORA_ALPHA}.\n"
            f"  Resuming it would train nothing. Fix: re-run the RUN CONFIG cell"
            f" (RUN is currently '{RUN}'), then the Drive cell, so CKPT_DIR changes"
            f" — or delete {CKPT_DIR}."
        )
    print(f"RESUMING {_last}  (r={_r}, alpha={_a}) — session-drop recovery")
else:
    print("FRESH training run ->", CKPT_DIR)

trainer.train(resume_from_checkpoint=_last)

# Save the adapter to Drive HERE, not in a later cell: a runtime recycle wipes
# /content and every local checkpoint with it (this has cost a full A100 run).
FINAL_ADAPTER = f"{DRIVE_DIR}/qwen3_4b_{RUN}_adapter"
model.save_pretrained(FINAL_ADAPTER); tokenizer.save_pretrained(FINAL_ADAPTER)
print("saved adapter ->", FINAL_ADAPTER)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


FRESH training run -> /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,550 | Num Epochs = 2 | Total steps = 1,320
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss
20,1.516691
40,0.508952
60,0.378887
80,0.338937
100,0.313353
120,0.308956
140,0.288456
160,0.294721
180,0.283611
200,0.283108


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-750/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-1250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_checkpoints/checkpoint-1320/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/arabicfc/qwen3_4b

saved adapter -> /content/drive/MyDrive/arabicfc/qwen3_4b_r16a16e2_adapter


In [ ]:
# 11) Inference helpers — load a checkpoint FROM DRIVE and predict (stops at end of call = fast)
import torch, gc
from unsloth import FastModel

def free_mem():
    for v in ("model","trainer"):
        if v in globals():
            try: del globals()[v]
            except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

def load_ckpt(path):
    """path = a LoRA checkpoint dir (checkpoint-XXX) OR a merged model dir. Unsloth reads
    adapter_config.json and pulls the base automatically for adapter checkpoints."""
    m, t = FastModel.from_pretrained(path, max_seq_length=max_seq_length,
                                     dtype=COMPUTE_DTYPE, load_in_4bit=not USE_BF16)
    FastModel.for_inference(m); t.padding_side = "left"
    return m, t

def run_inference(m, t, rows):
    prompts = [r["text"].split(SEP)[0] + SEP + "\n" for r in rows]
    BATCH, preds = 8, []
    for s in range(0, len(prompts), BATCH):
        chunk = prompts[s:s+BATCH]
        inp = t(text=chunk, return_tensors="pt", padding=True, truncation=True,
                max_length=max_seq_length, add_special_tokens=False).to(m.device)
        with torch.no_grad():
            gen = m.generate(**inp, max_new_tokens=300, do_sample=False,
                             pad_token_id=t.eos_token_id,
                             stop_strings=["<end_function_call>", "<end_of_turn>"], tokenizer=t)
        for j in range(len(chunk)):
            raw = t.decode(gen[j][inp["input_ids"].shape[1]:], skip_special_tokens=False)
            o = parse_model_output(raw)
            preds.append({"id": s+j, "tool_called": o["function_name"],
                          "arguments": o["arguments"], "think": o["think"]})
    return preds

print("helpers ready: load_ckpt(), run_inference(), free_mem()")

helpers ready: load_ckpt(), run_inference(), free_mem()


In [ ]:
# 12) PP v2 — enum-safe snap + validated ISO->Arabic date re-extraction + schema-type coercion
import re
from difflib import SequenceMatcher

# schema types {tool: {param: TYPE}} — every row offers the full 27-tool set, so any row has them all
TYPE = {}
for r in dev_raw:                      # small split; same 27-tool schema as train
    for t in r["tools"]:
        fn = t["function"]; d = TYPE.setdefault(fn["name"], {})
        for k, v in fn["parameters"]["properties"].items():
            if v is not None:
                d[k] = v["type"].upper()
    if len(TYPE) >= 27:
        break

# (a) snap free-text back to a verbatim query span — SKIP code/enum fields (gold there is a fixed code)
NO_SNAP = {"target_language","currency","from_currency","to_currency",
           "type","termination_type","search_type"}
def snap(pred_val, query, min_ratio=0.55):
    # compare on normalized text, but RETURN the original query span (keep hamza/diacritics that gold uses)
    p = norm(pred_val)
    if not p or p in norm(query): return pred_val
    words = query.split(); best, best_r = pred_val, min_ratio
    L = max(len(p.split()), 1)
    for w in range(max(1, L-2), L+3):
        for i in range(len(words)-w+1):
            cand = " ".join(words[i:i+w])
            r = SequenceMatcher(None, p, norm(cand)).ratio()
            if r > best_r: best, best_r = cand, r   # best = original surface span
    return best

# (b) ISO-date -> Arabic surface span pulled from the query (validated: +0 corruption on dev)
WD  = r"(?:السبت|الأحد|الاثنين|الإثنين|الثلاثاء|الأربعاء|الخميس|الجمعة)"
MO  = (r"(?:يناير|فبراير|مارس|أبريل|إبريل|مايو|يونيو|يوليو|أغسطس|سبتمبر|أكتوبر|نوفمبر|ديسمبر|"
       r"كانون الثاني|شباط|آذار|نيسان|أيار|حزيران|تموز|آب|أيلول|تشرين الأول|تشرين الثاني|كانون الأول)")
DIG = r"[0-9٠-٩]+"
_PATS = [re.compile(p) for p in [
    rf"يوم\s+{WD}", rf"{DIG}\s+{MO}", rf"{MO}\s+{DIG}",
    r"الأسبوع\s+القادم|الأسبوع\s+المقبل|نهاية\s+الأسبوع|بعد\s+غد|بعد\s+بكرة",
    WD, r"اليوم|غدًا|غدا|بكرة"]]
_ISO = re.compile(r"^\d{4}-\d{2}-\d{2}$")
DATE_FIELDS = ["check_in","check_out","departure_date","return_date","date"]
def _extract_dates(q):
    spans = []
    for pat in _PATS:
        for m in pat.finditer(q):
            s, e = m.span()
            if any(not (e <= os or s >= oe) for os, oe, _ in spans): continue
            spans.append((s, e, m.group(0).strip()))
    spans.sort(key=lambda x: x[0])
    return [t for _, _, t in spans]

# (c) schema-type coercion: STRING->str, NUMBER/INTEGER->float (gold convention, 0 exceptions)
_AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def _coerce(fn, k, v):
    st = TYPE.get(fn, {}).get(k)
    if v is None or v == "": return None, True
    if st == "STRING":
        if isinstance(v, bool):  return v, False
        if isinstance(v, int):   return str(v), False
        if isinstance(v, float): return (str(int(v)) if v == int(v) else str(v)), False
        return v, False
    if st in ("NUMBER", "INTEGER"):
        if isinstance(v, str):
            s = v.translate(_AR2EN).strip()
            try: return float(s), False
            except ValueError: return v, False
        if isinstance(v, int) and not isinstance(v, bool): return float(v), False
        return v, False
    return v, False

def postprocess_v2(fn, args, row_text):
    q = user_query(row_text); out = {}
    # 1) snap free-text (skip enum/code + date fields); then coerce types
    for k, v in args.items():
        if isinstance(v, str) and k not in NO_SNAP and k not in DATE_FIELDS:
            v = snap(v, q)
        nv, drop = _coerce(fn, k, v)
        if drop: continue
        out[k] = nv
    # 2) ISO date fields -> Arabic surface spans, assigned in query order
    dfields = [k for k in DATE_FIELDS if k in out and isinstance(out[k], str) and _ISO.match(out[k])]
    if dfields:
        phrases = _extract_dates(q)
        for j, k in enumerate([k for k in DATE_FIELDS if k in dfields]):
            if j < len(phrases): out[k] = phrases[j]
    return out

In [ ]:
# 13) Pick the BEST checkpoint from Drive — eval each on dev (RAW vs +PPv2), keep the winner
#     Scored with the OFFICIAL grader (cell 3): one score_full() per variant gives
#     FnAcc / ArgEM / Overall A / Overall B / per-dialect gaps in a single pass.
import os, gc, torch
free_mem()   # release the training model/optimizer before loading checkpoints

# Auto-discover this run's checkpoints — step numbers depend on batch size and
# EPOCHS, so hardcoding them breaks the moment someone changes the config.
import re
CANDIDATES = sorted((os.path.join(CKPT_DIR, d) for d in os.listdir(CKPT_DIR)
                     if re.fullmatch(r"checkpoint-\d+", d)),
                    key=lambda p: int(p.rsplit("-", 1)[1]))
if globals().get("FINAL_ADAPTER") and os.path.isdir(FINAL_ADAPTER):
    CANDIDATES.append(FINAL_ADAPTER)   # end-of-training state (past the last save_steps)
assert CANDIDATES, f"no checkpoints found under {CKPT_DIR}"
print("candidates:", [c.split('/')[-1] for c in CANDIDATES])

def eval_ckpt(path):
    m, t = load_ckpt(path)
    p = run_inference(m, t, dev_raw)
    p_pp = [{**x, "arguments": postprocess_v2(x["tool_called"], x["arguments"], r["text"])}
            for x, r in zip(p, dev_raw)]
    r0, r1 = score_full(p), score_full(p_pp)
    use_pp = r1["overall_a"] > r0["overall_a"]   # STRICT: RAW wins ties (PP has hurt Qwen before)
    win = r1 if use_pp else r0                   # every reported number comes from the SAME variant
    del m, t; gc.collect(); torch.cuda.empty_cache()
    return {"preds": p_pp if use_pp else p, "pp": use_pp, "rep": win,
            "OverallA": win["overall_a"], "OverallB": win["overall_b"],
            "ArgEM": win["argem"], "FnAcc": win["fnacc"],
            "raw_A": r0["overall_a"], "pp_A": r1["overall_a"]}

results = {}
for ck in CANDIDATES:
    print("evaluating", ck.split('/')[-1], "...")
    r = eval_ckpt(ck); results[ck] = r
    print(f"   RAW A {r['raw_A']:.4f} | +PPv2 A {r['pp_A']:.4f}  -> use {'PPv2' if r['pp'] else 'RAW'}"
          f"   (FnAcc {r['FnAcc']:.4f}, ArgEM {r['ArgEM']:.4f}, B {r['OverallB']:.4f})")

BEST_CKPT = max(results, key=lambda k: results[k]["OverallA"])
best = results[BEST_CKPT]["preds"]
print(f"\n>>> BEST = {BEST_CKPT.split('/')[-1]}  (PP={results[BEST_CKPT]['pp']})")
report(best, f"OFFICIAL dev — {BEST_CKPT.split('/')[-1]}")
print("\n    bar to beat: gemma-e2b official Overall A 0.9112 / ArgEM 0.852 (rank 17)")
print("    prior qwen ckpt-1320 official dev: A 0.9253 / ArgEM 0.8780")


candidates: ['checkpoint-1000', 'checkpoint-1250', 'checkpoint-1320']
evaluating checkpoint-1000 ...
==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

   RAW A 0.9244 | +PPv2 A 0.9148  -> use RAW   (FnAcc 1.0000, ArgEM 0.8740, B 0.9205)
evaluating checkpoint-1250 ...
==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

KeyboardInterrupt: 

In [ ]:
# 14) Error analysis on THIS model (per-dialect + per-function buckets)
from collections import Counter, defaultdict

per = defaultdict(lambda: [0,0])
fails = Counter()
for i, (p, g, r) in enumerate(zip(best, dev_gold, dev_raw)):
    if g["fn"] == "none": continue
    d = r["dialect"]
    per[d][1] += 1
    ga, pa = norm_args(g["args"]), norm_args(p["arguments"])
    ok = arg_ok(p, i)                      # official matcher -> buckets agree with score()
    if ok:
        per[d][0] += 1
        continue
    if p["tool_called"] != g["fn"]: cat = "wrong_fn"
    elif any(k not in pa for k in ga): cat = "missing_key"
    else: cat = "wrong_value"
    fails[(d, g["fn"], cat)] += 1

print("=== per-dialect ArgEM ===")
for d,(c,n) in sorted(per.items()):
    print(f"{d:12s} ArgEM {c/max(n,1):.3f}  (n={n})")
print("\n=== top failure buckets ===")
for k, n in fails.most_common(20): print(n, k)

NameError: name 'best' is not defined

In [ ]:
# 15) Write submissions
with open("submission_trackA.jsonl","w",encoding="utf-8") as f:
    for p in best:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"]}, ensure_ascii=False)+"\n")
with open("submission_trackB.jsonl","w",encoding="utf-8") as f:
    for p in best:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"],"think":p.get("think","")}, ensure_ascii=False)+"\n")
print("wrote submission_trackA.jsonl + submission_trackB.jsonl")

NameError: name 'best' is not defined

## 16) Save the fine-tuned model to Google Drive (persists across sessions)

In [ ]:
# 16) Save the BEST checkpoint as a merged 16-bit model to Drive (ready for the blind test)
import os, gc, torch
OUT = globals().get("DRIVE_DIR") or "/content/drive/MyDrive/arabicfc"
BEST_MERGED = f"{OUT}/qwen3_4b_{RUN}_best_merged"   # <- blind-test cell loads this
src = globals().get("BEST_CKPT")
assert src, "run the checkpoint-selection cell first (it sets BEST_CKPT)"
print("merging BEST checkpoint:", src.split('/')[-1], "->", BEST_MERGED)
m, t = load_ckpt(src)
m.save_pretrained_merged(BEST_MERGED, t, save_method="merged_16bit")
print("saved merged BEST ->", BEST_MERGED)
del m, t; gc.collect(); torch.cuda.empty_cache()

ModuleNotFoundError: No module named 'unsloth'

## 17) BLIND TEST SET — run this when the organizers release the test split

Fresh session? Run **cell 1 (install)**, the **RUN CONFIG** cell, **cell [splits/parser]**, and **cell [PP v2]** first (they define `prep`, `SEP`, `parse_model_output`, `user_query`, `postprocess_v2`, `norm`), then run this cell — it reloads the saved model from Drive (Colab) or /kaggle/working (Kaggle) on its own.

Assumes the test rows share the dev `text` format (tool declarations + user turn up to `<start_of_turn>model`). If organizers ship a different schema (e.g. raw `query` + `tools`), reformat into that same pre-`model` prompt string before generating.

In [ ]:
# 17) BLIND-TEST inference -> submission files
#     Loads ONE explicit checkpoint. No post-processing: PP is applied separately.
#     Additions vs the previous version:
#       1. <think>\n PRIMING     -> ThinkRate 0.889 -> ~1.0 = +0.0222 Overall B
#       2. schema-typed parser   -> stops int() eating leading zeros on ID fields
#       3. validity guards       -> 0 declarations => none ; tools offered => never none
#       4. dtype bug fixed       -> COMPUTE_DTYPE was computed then ignored
#       5. truncation counter    -> catches max_new_tokens being too small
import os, re, json, torch
from datasets import load_dataset
from unsloth import FastModel

# ---- what to run (edit these) ----------------------------------------------
CKPT   = "/content/drive/MyDrive/arabicfc/qwen3_4b_checkpoints/checkpoint-1320"
PRIME_THINK = True    # biggest single lever on Track B
GUARDS      = True    # forbid impossible outputs (costs one small extra pass)
BATCH, MAXNEW = 8, 300
SEP = "<start_of_turn>model"
# ----------------------------------------------------------------------------

assert os.path.isdir(CKPT), f"not found: {CKPT}  (ls the parent to see what exists)"
assert "prep" in globals(), "Run the definition cells first (missing prep)."

if "USE_BF16" not in globals():          # fresh session: derive instead of NameError
    from unsloth import is_bfloat16_supported
    USE_BF16 = is_bfloat16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("ckpt:", os.path.basename(CKPT), "| bf16:", USE_BF16,
      "| prime think:", PRIME_THINK, "| guards:", GUARDS)

# FIX 4: was hard-coded float16 while COMPUTE_DTYPE sat unused — on an A100 that
# silently discards bf16 and runs the model in a dtype it was not trained in.
model, tokenizer = FastModel.from_pretrained(CKPT, max_seq_length=2048,
                                             dtype=COMPUTE_DTYPE, load_in_4bit=False)
FastModel.for_inference(model)
tokenizer.padding_side = tokenizer.truncation_side = "left"
# left on BOTH: right-truncation would cut the trailing <start_of_turn>model marker

test_raw = load_dataset("TuwaiqAcademy/AISA-ArabicFC", split="test")
test_raw = test_raw.map(lambda r: {"text": prep(r["text"])})
assert "id" not in test_raw.column_names, "test gained an id column — use it instead of row order"
ids     = list(range(len(test_raw)))     # grader keys on enumerate index (cf. data_loader.py)
texts   = test_raw["text"]
assert all(t.count(SEP) == 1 for t in texts)

# ---- schema map: the parser needs it to know what to cast ------------------
TYPE = {}
for r in test_raw:
    for tool in r["tools"]:
        f = tool["function"]; d = TYPE.setdefault(f["name"], {})
        for k, v in (f["parameters"]["properties"] or {}).items():
            if v is not None:
                d[k] = str(v.get("type", "")).upper()

ID_FIELDS = {"account_number","iban","id_number","insurance_number","iqama_number",
             "national_id","passport_number","phone","phone_number",
             "recipient_iban","reference_number","visa_number"}
AR2EN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# ---- FIX 2: typed parser ---------------------------------------------------
def parse_typed(text, TYPE):
    # The old parser did `float(v) if "." in v else int(v)`, so a gold '00112233'
    # came back as 112233 and could never match. The schema decides the cast now.
    out = {"function_name": "none", "arguments": {}, "think": ""}
    m = re.search(r"<think>\s*(.*?)\s*</think>", text, re.DOTALL)
    if m: out["think"] = m.group(1).strip()
    m = re.search(r"<start_function_call>\s*call:(\w+)\{(.*?)\}\s*<end_function_call>",
                  text, re.DOTALL)
    if not m: return out
    fn = out["function_name"] = m.group(1)
    for key, sval, nval in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", m.group(2)):
        raw  = (sval if sval else nval).strip()
        want = TYPE.get(fn, {}).get(key)
        if key in ID_FIELDS or want == "STRING":
            val = raw                                   # leading zeros survive
        elif want in ("NUMBER", "INTEGER"):
            try: val = float(raw.translate(AR2EN))
            except ValueError: val = raw
        else:
            try: val = float(raw) if "." in raw else int(raw)
            except ValueError: val = raw
        out["arguments"][key] = val
    return out

# ---- FIX 1: prime the turn so the model cannot skip its reasoning ----------
SUFFIX  = "\n<think>\n" if PRIME_THINK else "\n"
prompts = [t.split(SEP)[0] + SEP + SUFFIX for t in texts]
DECLS   = [re.findall(r"<start_function_declaration>declaration:(\w+)\{", t.split(SEP)[0])
           for t in texts]

_lens = [len(tokenizer(text=p, add_special_tokens=False)["input_ids"]) for p in prompts]
assert max(_lens) <= 2048, f"prompt is {max(_lens)} tokens — raise max_seq_length, do NOT truncate"
print("test rows:", len(prompts), "| max prompt tokens:", max(_lens))

def generate(batch_prompts):
    inp = tokenizer(text=batch_prompts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=MAXNEW, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id,
                             stop_strings=["<end_function_call>", "<end_of_turn>"],
                             tokenizer=tokenizer)
    return [tokenizer.decode(gen[j][inp["input_ids"].shape[1]:], skip_special_tokens=False)
            for j in range(len(batch_prompts))]

preds, truncated = [], 0
for s in range(0, len(prompts), BATCH):
    chunk = prompts[s:s+BATCH]
    for j, raw in enumerate(generate(chunk)):
        i = s + j
        # the primed "<think>\n" is NOT in the generated text — put it back or the
        # regex finds no think at all and ThinkRate collapses to zero
        if PRIME_THINK: raw = "<think>\n" + raw
        if "<end_function_call>" not in raw and DECLS[i]: truncated += 1
        o = parse_typed(raw, TYPE)
        preds.append({"id": ids[i], "tool_called": o["function_name"],
                      "arguments": o["arguments"], "think": o["think"]})
    if s % (BATCH * 20) == 0:
        print(f"  {s + len(chunk)}/{len(prompts)}")
print("truncated (no <end_function_call> on a tool row):", truncated, "— want ~0")

# ---- validity audit (report only — the model already gets these right) -----
bad_abstain = sum(1 for i, p in enumerate(preds) if not DECLS[i] and p["tool_called"] != "none")
false_none  = sum(1 for i, p in enumerate(preds) if DECLS[i] and p["tool_called"] == "none")
bad_tool    = sum(1 for i, p in enumerate(preds)
                  if p["tool_called"] != "none" and p["tool_called"] not in DECLS[i])
print(f"audit: no-tool rows that called one {bad_abstain} | "
      f"tool rows answered none {false_none} | undeclared tool {bad_tool}  (all want 0)")
if bad_abstain or false_none or bad_tool:
    print("  ^ non-zero: this checkpoint is weaker than L16 — consider re-enabling the guards")

# ---- write -----------------------------------------------------------------
assert len(preds) == len(test_raw) == 1125
assert len({p["id"] for p in preds}) == len(preds)
n_think = sum(1 for p in preds if len(p["think"].strip()) > 5)
print("generated", len(preds),
      "| none:", sum(p["tool_called"] == "none" for p in preds),
      f"| ThinkRate: {n_think}/{len(preds)} = {n_think/len(preds):.4f}")
if n_think < len(preds):
    print(f"  -> Overall B is {0.20*(1-n_think/len(preds)):.4f} below its ceiling")

with open("test_submission_trackA.jsonl","w",encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"]}, ensure_ascii=False)+"\n")
with open("test_submission_trackB.jsonl","w",encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps({"id":p["id"],"tool_called":p["tool_called"],
                            "arguments":p["arguments"],"think":p.get("think","")}, ensure_ascii=False)+"\n")
print("wrote test_submission_trackA.jsonl + test_submission_trackB.jsonl")
